In [1]:
import pandas as pd
from pathlib import Path

df_cropyield = pd.read_pickle("data/raw/pickle/df_crop_yield_2016_2026.pkl")

# Optional: standardize column names
df_cropyield.columns = df_cropyield.columns.str.lower()

# View data
print(df_cropyield.shape)
df_cropyield.head()


(455359, 16)


,id,year,state_name,state_code,district_name,district_code,crop_name,crop_code,crop_type,season,area,area_unit,production,production_unit,yield,yield_unit
0,0,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Arhar/Tur,202.0,Pulses,Kharif,21400.0,Hectare,2600.0,Tonnes,0.121,Tonnes/Hectare
1,1,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Bajra,103.0,Cereals,Kharif,1400.0,Hectare,500.0,Tonnes,0.357,Tonnes/Hectare
2,2,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Castor Seed,1002.0,Oilseeds,Kharif,1000.0,Hectare,100.0,Tonnes,0.100,Tonnes/Hectare
3,3,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Cotton(Lint),1101.0,Fiber Crops,Kharif,7300.0,Hectare,9400.0,Tonnes,1.288,Tonnes/Hectare
4,4,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Dry Chillies,502.0,Spices,Kharif,3700.0,Hectare,7100.0,Tonnes,1.919,Tonnes/Hectare


In [ ]:
import pandas as pd

# Clean and create base year
df_cropyield["year_base"] = (
 df_cropyield["year"]
.astype(str)
.str.strip()
.str.split("-")
.str[0]
.astype(int)
)

# Filter rows for 2016 to 2022
df_cropyield_filtered = df_cropyield[
 df_cropyield["year_base"].between(2016, 2022)
].copy()

# Display result
print(df_cropyield_filtered.head())
print(df_cropyield_filtered.shape)

            id       year state_name  state_code district_name  district_code  \
132958  308685  2016-2017      Assam          18       Chirang            612   
133052  308843  2016-2017      Assam          18     Dibrugarh            286   
133125  308882  2016-2017      Assam          18      Goalpara            287   
133238  308981  2016-2017      Assam          18        Jorhat            290   
133717  309002  2016-2017      Assam          18        Kamrup            291   

       crop_name  crop_code crop_type season    area area_unit  production  \
132958      Gram      201.0    Pulses   Rabi   130.0   Hectare        61.0   
133052     Wheat      106.0   Cereals   Rabi     3.0   Hectare         4.0   
133125      Gram      201.0    Pulses   Rabi   165.0   Hectare       107.0   
133238    Masoor      205.0    Pulses   Rabi   509.0   Hectare       401.0   
133717  Arecanut      507.0    Spices   Rabi  3825.0   Hectare      4145.0   

       production_unit  yield      yield_uni

In [3]:
# Group by the specified columns and calculate the average yield
df_cropyield_grouped = df_cropyield_filtered.groupby(
    ["district_name", "crop_name", "season", "year_base"], 
    as_index=False
)["yield"].mean()
df_cropyield_grouped


,district_name,crop_name,season,year_base,yield
0,Adilabad,Arhar/Tur,Kharif,2016,1.104
1,Adilabad,Arhar/Tur,Kharif,2017,1.077
2,Adilabad,Arhar/Tur,Kharif,2018,1.331
3,Adilabad,Arhar/Tur,Kharif,2019,1.243
4,Adilabad,Arhar/Tur,Kharif,2020,0.907
...,...,...,...,...,...
147794,Zunheboto,Wheat,Rabi,2018,1.812
147795,Zunheboto,Wheat,Rabi,2019,1.813
147796,Zunheboto,Wheat,Rabi,2020,1.822
147797,Zunheboto,Wheat,Rabi,2021,1.801


In [4]:
# 1. Define the dictionary mapping synonyms to standard names
season_unification_map = {
    'Kharif': 'Kharif',
    'Autumn': 'Kharif',
    'Rabi': 'Rabi',
    'Winter': 'Rabi',
    'Summer': 'Summer',
    'Whole Year': 'Whole Year',
    'Total': 'Whole Year'
}

# 2. Map and update the season column directly
df_cropyield_grouped['season'] = df_cropyield_grouped['season'].map(season_unification_map)


In [5]:
df_cropyield_grouped["season"].unique()

array(['Kharif', 'Rabi', 'Whole Year', 'Summer'], dtype=object)

In [6]:
# 1. Define the mapping of seasons to their respective month numbers
season_to_months = {
    'Kharif': [7, 8, 9, 10],
    'Rabi': [10, 11, 12, 1, 2, 3],
    'Summer': [3, 4, 5, 6],
    'Whole Year': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
}

# 2. Create the "month" column using map
df_cropyield_grouped['month'] = df_cropyield_grouped['season'].map(season_to_months)
df_cropyield_grouped

,district_name,crop_name,season,year_base,yield,month
0,Adilabad,Arhar/Tur,Kharif,2016,1.104,"[7, 8, 9, 10]"
1,Adilabad,Arhar/Tur,Kharif,2017,1.077,"[7, 8, 9, 10]"
2,Adilabad,Arhar/Tur,Kharif,2018,1.331,"[7, 8, 9, 10]"
3,Adilabad,Arhar/Tur,Kharif,2019,1.243,"[7, 8, 9, 10]"
4,Adilabad,Arhar/Tur,Kharif,2020,0.907,"[7, 8, 9, 10]"
...,...,...,...,...,...,...
147794,Zunheboto,Wheat,Rabi,2018,1.812,"[10, 11, 12, 1, 2, 3]"
147795,Zunheboto,Wheat,Rabi,2019,1.813,"[10, 11, 12, 1, 2, 3]"
147796,Zunheboto,Wheat,Rabi,2020,1.822,"[10, 11, 12, 1, 2, 3]"
147797,Zunheboto,Wheat,Rabi,2021,1.801,"[10, 11, 12, 1, 2, 3]"


In [7]:
df_cropyield_grouped = df_cropyield_grouped.rename(columns={"district_name": "district"})


In [8]:
import pandas as pd
from pathlib import Path

df_precipitation = pd.read_pickle("data/processed/precipitation3.pkl")

# Optional: standardize column names
df_precipitation.columns = df_precipitation.columns.str.lower()

# View data
print(df_precipitation.head())
print(df_precipitation.shape)

               date        state          district  precipitation_mm  \
1400000  2017-12-16  West Bengal  Dakshin Dinajpur              0.00   
1400001  2017-12-17  West Bengal  Dakshin Dinajpur              0.00   
1400002  2017-12-18  West Bengal  Dakshin Dinajpur              0.00   
1400003  2017-12-19  West Bengal  Dakshin Dinajpur              1.17   
1400004  2017-12-20  West Bengal  Dakshin Dinajpur              0.00   

              date  
1400000 2017-12-16  
1400001 2017-12-17  
1400002 2017-12-18  
1400003 2017-12-19  
1400004 2017-12-20  
(758332, 5)


In [9]:
import pandas as pd

# 1. Clean up duplicate columns by keeping only the first instance
df_precipitation = df_precipitation.loc[:, ~df_precipitation.columns.duplicated()]

# 2. Now convert the single "date" column safely
df_precipitation["date"] = pd.to_datetime(df_precipitation["date"])

# 3. Extract year and month
df_precipitation["year"] = df_precipitation["date"].dt.year
df_precipitation["month"] = df_precipitation["date"].dt.month

# 4. Filter East Godavari data for years 2016 to 2022
df_precipitation_filtered = df_precipitation[
    df_precipitation["year"].between(2016, 2022)
].copy()

# View filtered dataframe
print(df_precipitation_filtered.head())
print(df_precipitation_filtered.shape)


              date        state          district  precipitation_mm  year  \
1400000 2017-12-16  West Bengal  Dakshin Dinajpur              0.00  2017   
1400001 2017-12-17  West Bengal  Dakshin Dinajpur              0.00  2017   
1400002 2017-12-18  West Bengal  Dakshin Dinajpur              0.00  2017   
1400003 2017-12-19  West Bengal  Dakshin Dinajpur              1.17  2017   
1400004 2017-12-20  West Bengal  Dakshin Dinajpur              0.00  2017   

         month  
1400000     12  
1400001     12  
1400002     12  
1400003     12  
1400004     12  
(758332, 6)


In [10]:
# Group by district, year (with trailing space), and month, then calculate the mean of precipitation_mm
df_precipitation_grouped = df_precipitation_filtered.groupby(
    ['district', 'year', 'month'], 
    as_index=False
)['precipitation_mm'].mean()

df_precipitation_grouped = df_precipitation_grouped.dropna(subset=["precipitation_mm"])


df_precipitation_grouped


C:\Users\u242122\AppData\Local\Temp\ipykernel_7324\3805268555.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_precipitation_grouped = df_precipitation_filtered.groupby(


,district,year,month,precipitation_mm
5,Agra,2016,6,2.351111
6,Agra,2016,7,14.222258
7,Agra,2016,8,9.511075
8,Agra,2016,9,4.651111
9,Agra,2016,10,1.188495
...,...,...,...,...
34848,Zunheboto,2022,1,0.814839
34849,Zunheboto,2022,2,1.260357
34850,Zunheboto,2022,3,1.711290
34851,Zunheboto,2022,4,4.574333


In [11]:
df_precipitation_grouped["precipitation_mm"].isna().sum()

np.int64(0)

In [12]:
df_cropyield_grouped

,district,crop_name,season,year_base,yield,month
0,Adilabad,Arhar/Tur,Kharif,2016,1.104,"[7, 8, 9, 10]"
1,Adilabad,Arhar/Tur,Kharif,2017,1.077,"[7, 8, 9, 10]"
2,Adilabad,Arhar/Tur,Kharif,2018,1.331,"[7, 8, 9, 10]"
3,Adilabad,Arhar/Tur,Kharif,2019,1.243,"[7, 8, 9, 10]"
4,Adilabad,Arhar/Tur,Kharif,2020,0.907,"[7, 8, 9, 10]"
...,...,...,...,...,...,...
147794,Zunheboto,Wheat,Rabi,2018,1.812,"[10, 11, 12, 1, 2, 3]"
147795,Zunheboto,Wheat,Rabi,2019,1.813,"[10, 11, 12, 1, 2, 3]"
147796,Zunheboto,Wheat,Rabi,2020,1.822,"[10, 11, 12, 1, 2, 3]"
147797,Zunheboto,Wheat,Rabi,2021,1.801,"[10, 11, 12, 1, 2, 3]"


In [13]:
import pandas as pd

# 1. Create a safe row tracker column from the index (avoiding .reset_index() entirely)
df_cropyield_grouped['temp_row_id'] = df_cropyield_grouped.index

# 2. Explode the "month" column so each month in the list gets its own row
exploded = df_cropyield_grouped.explode('month')
exploded['month'] = exploded['month'].astype(int)

# 3. Calculate the actual calendar year (calendar_year) for Rabi and Summer months
def get_calendar_year(row):
    if row['season'] == 'Rabi' and row['month'] in [1, 2, 3]:
        return row['year_base'] + 1
    elif row['season'] == 'Summer' and row['month'] in [3, 4, 5, 6]:
        return row['year_base'] + 1
    return row['year_base']

exploded['calendar_year'] = exploded.apply(get_calendar_year, axis=1)

# 4. Merge the exploded table with df_precipitation_grouped
merged = pd.merge(
    exploded,
    df_precipitation_grouped,
    left_on=['district', 'calendar_year', 'month'],
    right_on=['district', 'year', 'month'],
    how='left'
)

# 5. Group back by our safe custom row ID and compute the average precipitation
avg_ppt_series = merged.groupby('temp_row_id')['precipitation_mm'].mean()

# 6. Assign the average precipitation back to your main DataFrame and clean up the tracker
df_cropyield_grouped['avg_ppt'] = df_cropyield_grouped['temp_row_id'].map(avg_ppt_series)
df_cropyield_grouped.drop(columns=['temp_row_id'], inplace=True)

df_cropyield_grouped


,district,crop_name,season,year_base,yield,month,avg_ppt
0,Adilabad,Arhar/Tur,Kharif,2016,1.104,"[7, 8, 9, 10]",NaN
1,Adilabad,Arhar/Tur,Kharif,2017,1.077,"[7, 8, 9, 10]",NaN
2,Adilabad,Arhar/Tur,Kharif,2018,1.331,"[7, 8, 9, 10]",NaN
3,Adilabad,Arhar/Tur,Kharif,2019,1.243,"[7, 8, 9, 10]",NaN
4,Adilabad,Arhar/Tur,Kharif,2020,0.907,"[7, 8, 9, 10]",NaN
...,...,...,...,...,...,...,...
147794,Zunheboto,Wheat,Rabi,2018,1.812,"[10, 11, 12, 1, 2, 3]",0.740824
147795,Zunheboto,Wheat,Rabi,2019,1.813,"[10, 11, 12, 1, 2, 3]",0.862141
147796,Zunheboto,Wheat,Rabi,2020,1.822,"[10, 11, 12, 1, 2, 3]",0.815841
147797,Zunheboto,Wheat,Rabi,2021,1.801,"[10, 11, 12, 1, 2, 3]",1.030815


In [14]:
df_cropyield_grouped["avg_ppt"].isna().sum()

np.int64(93324)

In [15]:
# Create df_cropyield_grouped_clean by removing NaN values from the avg_ppt column
df_cropyield_grouped_clean = df_cropyield_grouped.dropna(subset=['avg_ppt']).copy()
df_cropyield_grouped_clean

,district,crop_name,season,year_base,yield,month,avg_ppt
497,Agra,Arhar/Tur,Kharif,2016,0.900,"[7, 8, 9, 10]",7.393235
498,Agra,Arhar/Tur,Kharif,2017,1.238,"[7, 8, 9, 10]",5.971496
504,Agra,Bajra,Kharif,2016,2.449,"[7, 8, 9, 10]",7.393235
505,Agra,Bajra,Kharif,2017,2.381,"[7, 8, 9, 10]",5.971496
512,Agra,Banana,Summer,2018,45.250,"[3, 4, 5, 6]",0.398736
...,...,...,...,...,...,...,...
147793,Zunheboto,Wheat,Rabi,2017,1.844,"[10, 11, 12, 1, 2, 3]",3.578312
147794,Zunheboto,Wheat,Rabi,2018,1.812,"[10, 11, 12, 1, 2, 3]",0.740824
147795,Zunheboto,Wheat,Rabi,2019,1.813,"[10, 11, 12, 1, 2, 3]",0.862141
147796,Zunheboto,Wheat,Rabi,2020,1.822,"[10, 11, 12, 1, 2, 3]",0.815841


In [16]:
df_cropyield_grouped_clean["crop_name"] = (
    df_cropyield_grouped_clean["crop_name"]
    .astype(str)
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

df_cropyield_grouped_clean

,district,crop_name,season,year_base,yield,month,avg_ppt
497,Agra,arhar/tur,Kharif,2016,0.900,"[7, 8, 9, 10]",7.393235
498,Agra,arhar/tur,Kharif,2017,1.238,"[7, 8, 9, 10]",5.971496
504,Agra,bajra,Kharif,2016,2.449,"[7, 8, 9, 10]",7.393235
505,Agra,bajra,Kharif,2017,2.381,"[7, 8, 9, 10]",5.971496
512,Agra,banana,Summer,2018,45.250,"[3, 4, 5, 6]",0.398736
...,...,...,...,...,...,...,...
147793,Zunheboto,wheat,Rabi,2017,1.844,"[10, 11, 12, 1, 2, 3]",3.578312
147794,Zunheboto,wheat,Rabi,2018,1.812,"[10, 11, 12, 1, 2, 3]",0.740824
147795,Zunheboto,wheat,Rabi,2019,1.813,"[10, 11, 12, 1, 2, 3]",0.862141
147796,Zunheboto,wheat,Rabi,2020,1.822,"[10, 11, 12, 1, 2, 3]",0.815841


In [17]:
df_cropyield_grouped_clean["crop_name"].unique()

array(['arhar/tur', 'bajra', 'banana', 'barley', 'coriander',
       'cotton(lint)', 'dry_chillies', 'garlic', 'gram', 'guar_seed',
       'jowar', 'maize', 'masoor', 'moong(green_gram)', 'onion',
       'peas_&_beans_(pulses)', 'potato', 'rapeseed_&mustard', 'rice',
       'sannhamp', 'sesamum', 'sugarcane', 'sweet_potato', 'urad',
       'wheat', 'cowpea(lobia)', 'other_kharif_pulses',
       'other_rabi_pulses', 'soyabean', 'tapioca', 'tobacco',
       'castor_seed', 'ginger', 'groundnut', 'linseed', 'moth',
       'other_oilseeds', 'other_cereals', 'other_summer_pulses',
       'safflower', 'sunflower', 'arecanut', 'black_pepper', 'cashewnut',
       'coconut', 'ragi', 'turmeric', 'small_millets', 'horse-gram',
       'niger_seed', 'jute', 'khesari', 'mesta', 'cardamom', 'dry_ginger',
       'arcanut_(processed)', 'cond-spcs_other', 'mango',
       'other_fresh_fruits', 'other_vegetables', 'pineapple'],
      dtype=object)

In [18]:
import pandas as pd

# Direct dictionary mapping: Crop Name -> Avg Net Profit (INR per Ton)
# (1 Ton = 10 Quintals)
crop_profit_per_ton = {
    'arhar/tur': 8500,
    'bajra': 3500,
    'banana': 12000,
    'barley': 4000,
    'coriander': 25000,
    'cotton(lint)': 8000,
    'dry_chillies': 60000,
    'garlic': 35000,
    'gram': 7500,
    'guar_seed': 9000,
    'jowar': 3800,
    'maize': 4500,
    'masoor': 8000,
    'moong(green_gram)': 9000,
    'onion': 4500,
    'peas_&_beans_(pulses)': 9500,
    'potato': 5000,
    'rapeseed_&mustard': 12000,
    'rice': 3500,
    'sannhamp': 4000,
    'sesamum': 15000,
    'sugarcane': 1350,  # Highly productive by tonnage, lower profit margin per individual ton
    'sweet_potato': 6000,
    'urad': 8500,
    'wheat': 5500,
    'cowpea(lobia)': 7500,
    'other_kharif_pulses': 6500,
    'other_rabi_pulses': 7000,
    'soyabean': 8000,
    'tapioca': 4500,
    'tobacco': 22000,
    'castor_seed': 9500,
    'ginger': 45000,
    'groundnut': 11000,
    'linseed': 8500,
    'moth': 7000,
    'other_oilseeds': 8000,
    'other_cereals': 3200,
    'other_summer_pulses': 7500,
    'safflower': 7500,
    'sunflower': 9000,
    'arecanut': 85000,
    'black_pepper': 120000,  # High-value export spice
    'cashewnut': 75000,
    'coconut': 15000,
    'ragi': 4000,
    'turmeric': 35000,
    'small_millets': 3500,
    'horse-gram': 6000,
    'niger_seed': 8500,
    'jute': 7000,
    'khesari': 5500,
    'mesta': 6000,
    'cardamom': 250000,      # Premium luxury crop
    'dry_ginger': 55000,
    'arcanut_(processed)': 110000,
    'cond-spcs_other': 25000,
    'mango': 25000,
    'other_fresh_fruits': 20000,
    'other_vegetables': 8000,
    'pineapple': 15000
}

# Clean backslashes from crop_name column if present in the data (e.g. 'peas\_&\_beans')
df_cropyield_grouped_clean['crop_name'] = df_cropyield_grouped_clean['crop_name'].str.replace('\\', '', regex=False)

# Map values directly to a new column
df_cropyield_grouped_clean['profit_per_ton_inr'] = df_cropyield_grouped_clean['crop_name'].map(crop_profit_per_ton)
df_cropyield_grouped_clean['profit']=df_cropyield_grouped_clean['profit_per_ton_inr']*df_cropyield_grouped_clean['yield']
df_cropyield_grouped_clean

,district,crop_name,season,year_base,yield,month,avg_ppt,profit_per_ton_inr,profit
497,Agra,arhar/tur,Kharif,2016,0.900,"[7, 8, 9, 10]",7.393235,8500,7650.0
498,Agra,arhar/tur,Kharif,2017,1.238,"[7, 8, 9, 10]",5.971496,8500,10523.0
504,Agra,bajra,Kharif,2016,2.449,"[7, 8, 9, 10]",7.393235,3500,8571.5
505,Agra,bajra,Kharif,2017,2.381,"[7, 8, 9, 10]",5.971496,3500,8333.5
512,Agra,banana,Summer,2018,45.250,"[3, 4, 5, 6]",0.398736,12000,543000.0
...,...,...,...,...,...,...,...,...,...
147793,Zunheboto,wheat,Rabi,2017,1.844,"[10, 11, 12, 1, 2, 3]",3.578312,5500,10142.0
147794,Zunheboto,wheat,Rabi,2018,1.812,"[10, 11, 12, 1, 2, 3]",0.740824,5500,9966.0
147795,Zunheboto,wheat,Rabi,2019,1.813,"[10, 11, 12, 1, 2, 3]",0.862141,5500,9971.5
147796,Zunheboto,wheat,Rabi,2020,1.822,"[10, 11, 12, 1, 2, 3]",0.815841,5500,10021.0


In [19]:

df_clean = df_cropyield_grouped_clean[
    (df_cropyield_grouped_clean["district"] == "Agra") & 
    (df_cropyield_grouped_clean["season"] == "Kharif")
]
df_clean

,district,crop_name,season,year_base,yield,month,avg_ppt,profit_per_ton_inr,profit
497,Agra,arhar/tur,Kharif,2016,0.900,"[7, 8, 9, 10]",7.393235,8500,7650.00
498,Agra,arhar/tur,Kharif,2017,1.238,"[7, 8, 9, 10]",5.971496,8500,10523.00
504,Agra,bajra,Kharif,2016,2.449,"[7, 8, 9, 10]",7.393235,3500,8571.50
505,Agra,bajra,Kharif,2017,2.381,"[7, 8, 9, 10]",5.971496,3500,8333.50
528,Agra,cotton(lint),Kharif,2016,0.349,"[7, 8, 9, 10]",7.393235,8000,2792.00
529,Agra,cotton(lint),Kharif,2017,0.249,"[7, 8, 9, 10]",5.971496,8000,1992.00
535,Agra,dry_chillies,Kharif,2016,0.922,"[7, 8, 9, 10]",7.393235,60000,55320.00
536,Agra,dry_chillies,Kharif,2017,0.923,"[7, 8, 9, 10]",5.971496,60000,55380.00
556,Agra,guar_seed,Kharif,2016,0.715,"[7, 8, 9, 10]",7.393235,9000,6435.00
557,Agra,guar_seed,Kharif,2017,0.716,"[7, 8, 9, 10]",5.971496,9000,6444.00


In [20]:
# import pandas as pd

# # Define a function to calculate the mean precipitation and find the crop with max profit
# def calculate_subgroup_metrics(group):
#     # Calculate average of the avg_ppt column
#     avg_avg_ppt = group['avg_ppt'].mean()
    
#     # Find the crop_name of the row containing the maximum value in the 'profit' column
#     if not group['profit'].dropna().empty:
#         idx_max = group['profit'].idxmax()
#         best_crop = group.loc[idx_max, 'crop_name']
#     else:
#         best_crop = None  # Handle edge cases with empty or all-NaN profit groups
        
#     return pd.Series({
#         'avg_avg_ppt': avg_avg_ppt,
#         'best_crop': best_crop
#     })

# # Group by district and season, and apply the calculation
# df_district_season_summary = (
#     df_cropyield_grouped_clean
#     .groupby(['district', 'season'])
#     .apply(calculate_subgroup_metrics)
#     .reset_index()
# )
# df_district_season_summary

In [21]:
import pandas as pd

# Define a function to filter rows in the top 20% of profit for each subgroup
def filter_top_percentile_crops(group, percentile_threshold=0.80):
    valid_profits = group['profit'].dropna()
    if not valid_profits.empty:
        # Calculate the 80th percentile threshold of profit for this specific group
        threshold = valid_profits.quantile(percentile_threshold)
        # Keep rows that are at or above the 80th percentile
        return group[group['profit'] >= threshold]
    else:
        return pd.DataFrame(columns=group.columns)

# Group by district and season, apply the filter, and reset the index
df_district_season_summary = (
    df_cropyield_grouped_clean
    .groupby(['district', 'season'], group_keys=False)
    .apply(filter_top_percentile_crops)
    .reset_index(drop=True)
)
df_district_season_summary

C:\Users\u242122\AppData\Local\Temp\ipykernel_7324\3003642477.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(filter_top_percentile_crops)


,district,crop_name,season,year_base,yield,month,avg_ppt,profit_per_ton_inr,profit
0,Agra,dry_chillies,Kharif,2016,0.922,"[7, 8, 9, 10]",7.393235,60000,55320.00
1,Agra,dry_chillies,Kharif,2017,0.923,"[7, 8, 9, 10]",5.971496,60000,55380.00
2,Agra,sugarcane,Kharif,2016,53.601,"[7, 8, 9, 10]",7.393235,1350,72361.35
3,Agra,sugarcane,Kharif,2017,56.684,"[7, 8, 9, 10]",5.971496,1350,76523.40
4,Agra,sweet_potato,Kharif,2016,11.166,"[7, 8, 9, 10]",7.393235,6000,66996.00
...,...,...,...,...,...,...,...,...,...
11567,Zunheboto,peas_&_beans_(pulses),Whole Year,2016,1.165,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]",6.379825,9500,11067.50
11568,Zunheboto,peas_&_beans_(pulses),Whole Year,2017,1.165,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]",6.183374,9500,11067.50
11569,Zunheboto,peas_&_beans_(pulses),Whole Year,2018,1.165,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]",0.315806,9500,11067.50
11570,Zunheboto,sugarcane,Whole Year,2021,12.300,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]",2.367450,1350,16605.00


In [22]:

df_clean1 = df_district_season_summary[
    (df_district_season_summary["district"] == "Agra") & 
    (df_district_season_summary["season"] == "Kharif")
]
df_clean1

,district,crop_name,season,year_base,yield,month,avg_ppt,profit_per_ton_inr,profit
0,Agra,dry_chillies,Kharif,2016,0.922,"[7, 8, 9, 10]",7.393235,60000,55320.00
1,Agra,dry_chillies,Kharif,2017,0.923,"[7, 8, 9, 10]",5.971496,60000,55380.00
2,Agra,sugarcane,Kharif,2016,53.601,"[7, 8, 9, 10]",7.393235,1350,72361.35
3,Agra,sugarcane,Kharif,2017,56.684,"[7, 8, 9, 10]",5.971496,1350,76523.40
4,Agra,sweet_potato,Kharif,2016,11.166,"[7, 8, 9, 10]",7.393235,6000,66996.00
5,Agra,sweet_potato,Kharif,2017,11.165,"[7, 8, 9, 10]",5.971496,6000,66990.00


In [23]:
import pandas as pd
import numpy as np

# Find the index of the crop with the maximum profit in each district-season-year group
idx_best = (
    df_cropyield_grouped_clean
    .groupby(['district', 'season', 'year_base'])['profit']
    .idxmax()
)

# Extract only these best rows to serve as our ML training dataset
df_ml = df_cropyield_grouped_clean.loc[idx_best].copy()

# Rename crop_name to represent our target label: 'best_crop'
df_ml.rename(columns={'crop_name': 'best_crop'}, inplace=True)

# Print unique best crops and count of samples to verify progress
print(f"Dataset prepared! Total training rows: {len(df_ml)}")
print(f"Unique recommended crops: {df_ml['best_crop'].nunique()}")
print(df_ml['best_crop'].value_counts().head(5))


Dataset prepared! Total training rows: 6895
Unique recommended crops: 47
best_crop
onion        746
potato       685
sugarcane    658
coconut      537
garlic       534
Name: count, dtype: int64


In [24]:
from sklearn.preprocessing import LabelEncoder

# 1. Create a dictionary to hold the encoders for features (district and season)
label_encoders = {}

# 2. Encode the feature columns
for col in ['district', 'season']:
    le = LabelEncoder()
    # Fit the encoder to the data and transform strings into numbers
    df_ml[col + '_encoded'] = le.fit_transform(df_ml[col].astype(str))
    # Save the encoder for making future predictions
    label_encoders[col] = le

# 3. Encode the target column (best_crop)
target_encoder = LabelEncoder()
df_ml['best_crop_encoded'] = target_encoder.fit_transform(df_ml['best_crop'].astype(str))

# Print a preview to see the mappings
print("Feature and target columns successfully encoded!\n")
print("District mappings (first 5):")
print(dict(zip(label_encoders['district'].classes_[:5], range(5))))
print("\nSeason mappings:")
print(dict(zip(label_encoders['season'].classes_, range(len(label_encoders['season'].classes_)))))
print("\nCrop target mappings (first 5):")
print(dict(zip(target_encoder.classes_[:5], range(5))))


Feature and target columns successfully encoded!

District mappings (first 5):
{'Agra': 0, 'Aizawl': 1, 'Ajmer': 2, 'Akola': 3, 'Alappuzha': 4}

Season mappings:
{'Kharif': 0, 'Rabi': 1, 'Summer': 2, 'Whole Year': 3}

Crop target mappings (first 5):
{'arecanut': 0, 'arhar/tur': 1, 'bajra': 2, 'banana': 3, 'barley': 4}


In [25]:
from sklearn.model_selection import train_test_split

# 1. Define our features (X) and target variable (y)
X = df_ml[['district_encoded', 'season_encoded', 'avg_ppt']]
y = df_ml['best_crop_encoded']

# 2. First split: Separate 70% of the dataset for training, leaving 30% temporary set
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# 3. Second split: Split the 30% temporary set equally (50-50) into Validation and Test sets
# This mathematically yields: Train (70%), Validation (15%), Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

# 4. Verify the split proportions
print("----------------------------------------")
print(f"Total dataset size:      {len(df_ml)} rows")
print(f"Training set size (70%): {len(X_train)} rows")
print(f"Validation set size (15%): {len(X_val)} rows")
print(f"Testing set size (15%):   {len(X_test)} rows")
print("----------------------------------------")


----------------------------------------
Total dataset size:      6895 rows
Training set size (70%): 4826 rows
Validation set size (15%): 1034 rows
Testing set size (15%):   1035 rows
----------------------------------------


In [26]:
from sklearn.ensemble import RandomForestClassifier

# 1. Initialize the Random Forest Classifier
# n_estimators=100 means we train 100 decision trees
# random_state=42 ensures that the results are reproducible
model = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Train (Fit) the model on our training data
print("Training the Random Forest model on the training set...")
model.fit(X_train, y_train)
print("Model training complete! 🎉")


Training the Random Forest model on the training set...
Model training complete! 🎉


In [27]:
# import numpy as np
# from sklearn.metrics import accuracy_score, classification_report

# # 1. Generate standard predictions and probabilities
# val_predictions = model.predict(X_val)
# test_predictions = model.predict(X_test)

# val_probabilities = model.predict_proba(X_val)
# test_probabilities = model.predict_proba(X_test)

# # 2. Get standard Top-1 accuracies
# val_accuracy_top1 = accuracy_score(y_val, val_predictions)
# test_accuracy_top1 = accuracy_score(y_test, test_predictions)


# # 3. Robust Manual Top-K Accuracy function (prevents unseen label errors)
# def calculate_manual_top_k(y_true, probabilities, model_classes, k=3):
#     correct_predictions = 0
#     y_true_array = np.array(y_true)
    
#     for i, true_label in enumerate(y_true_array):
#         # Sort probabilities to get the indices of the top-k highest probabilities
#         top_k_indices = np.argsort(probabilities[i])[-k:]
#         # Map those indices back to the actual trained crop class IDs
#         top_k_crop_classes = model_classes[top_k_indices]
        
#         # If the actual crop is in the top-k recommendations, count as correct
#         if true_label in top_k_crop_classes:
#             correct_predictions += 1
            
#     return correct_predictions / len(y_true)


# # 4. Compute robust Top-3 and Top-5 accuracies
# test_accuracy_top3 = calculate_manual_top_k(y_test, test_probabilities, model.classes_, k=3)
# test_accuracy_top5 = calculate_manual_top_k(y_test, test_probabilities, model.classes_, k=5)


# print("==================================================")
# print(f"Standard Top-1 Accuracy (Validation): {val_accuracy_top1 * 100:.2f}%")
# print(f"Standard Top-1 Accuracy (Testing):    {test_accuracy_top1 * 100:.2f}%")
# print("--------------------------------------------------")
# print(f"⭐ Top-3 Recommendation Accuracy:     {test_accuracy_top3 * 100:.2f}%")
# print(f"⭐ Top-5 Recommendation Accuracy:     {test_accuracy_top5 * 100:.2f}%")
# print("==================================================")


# # 5. Print standard classification report cleanly
# classes_in_test = sorted(list(set(y_test) | set(test_predictions)))
# crop_names_in_test = target_encoder.inverse_transform(classes_in_test)

# print("\nDetailed Test Set Classification Report (Single-Crop Baseline):\n")
# print(classification_report(
#     y_test, 
#     test_predictions, 
#     labels=classes_in_test,
#     target_names=crop_names_in_test, 
#     zero_division=0
# ))


In [30]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# 1. Expand display limits so no rows are hidden in Jupyter/Console
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)
np.set_printoptions(threshold=np.inf)

# 2. Generate predictions and probabilities for all sets
val_predictions = model.predict(X_val)
test_predictions = model.predict(X_test)

val_probabilities = model.predict_proba(X_val)
test_probabilities = model.predict_proba(X_test)

# 3. Compute Top-1 accuracies
val_accuracy_top1 = accuracy_score(y_val, val_predictions)
test_accuracy_top1 = accuracy_score(y_test, test_predictions)

# 4. Robust Manual Top-K Accuracy calculation function
def calculate_manual_top_k(y_true, probabilities, model_classes, k=3):
    correct_predictions = 0
    y_true_array = np.array(y_true)
    for i, true_label in enumerate(y_true_array):
        top_k_indices = np.argsort(probabilities[i])[-k:]
        top_k_crop_classes = model_classes[top_k_indices]
        if true_label in top_k_crop_classes:
            correct_predictions += 1
    return correct_predictions / len(y_true)

# 5. Compute Top-3 and Top-5 accuracies on Testing Set
test_accuracy_top3 = calculate_manual_top_k(y_test, test_probabilities, model.classes_, k=3)
test_accuracy_top5 = calculate_manual_top_k(y_test, test_probabilities, model.classes_, k=5)

# 6. Print general accuracy metrics
print("==================================================")
print(f"Standard Top-1 Accuracy (Validation): {val_accuracy_top1 * 100:.2f}%")
print(f"Standard Top-1 Accuracy (Testing):    {test_accuracy_top1 * 100:.2f}%")
print("--------------------------------------------------")
print(f"⭐ Top-3 Recommendation Accuracy:     {test_accuracy_top3 * 100:.2f}%")
print(f"⭐ Top-5 Recommendation Accuracy:     {test_accuracy_top5 * 100:.2f}%")
print("==================================================")

# 7. Print the Detailed Classification Report ONLY for the Testing Set
# Dynamically extract classes present in this subset to avoid ValueErrors
classes_in_test = sorted(list(set(y_test) | set(test_predictions)))
crop_names_in_test = target_encoder.inverse_transform(classes_in_test)

print("\nDetailed Classification Report - TESTING Set:")
print("-" * 70)
print(classification_report(
    y_test, 
    test_predictions, 
    labels=classes_in_test,
    target_names=crop_names_in_test, 
    zero_division=0
))


Standard Top-1 Accuracy (Validation): 41.10%
Standard Top-1 Accuracy (Testing):    38.16%
--------------------------------------------------
⭐ Top-3 Recommendation Accuracy:     66.38%
⭐ Top-5 Recommendation Accuracy:     80.58%

Detailed Classification Report - TESTING Set:
----------------------------------------------------------------------
                       precision    recall  f1-score   support

             arecanut       0.35      0.32      0.33        22
            arhar/tur       0.10      0.10      0.10        10
                bajra       0.00      0.00      0.00         1
               banana       0.20      0.22      0.21        41
               barley       0.33      0.33      0.33         6
         black_pepper       0.00      0.00      0.00         1
            cashewnut       0.00      0.00      0.00         2
          castor_seed       0.00      0.00      0.00         1
              coconut       0.46      0.71      0.56        65
            coriander 

In [1]:
df_ml

NameError: name 'df_ml' is not defined